In [1]:
from torchvision.datasets import MNIST, EMNIST, KMNIST
import torch
from torch import nn
from torchvision import transforms
from tqdm import tqdm
import copy
from torch.utils.data import Subset
import torch.nn.functional as F
# CIFAR100 and ImageNet-100/1000 are also common benchmarks in Class-Incremental Learning
'''
We can do:
- base CIL
- layer old networks with new ones(meaning the network grows
and computes a value for how much each independent network
contributes to the output, using some kind of final head)
- CIL/DIL mix, by retraining on old classes while adding new ones(issue: old digits will be higher weighted)

'''

'\nWe can do:\n- base CIL\n- layer old networks with new ones(meaning the network grows\nand computes a value for how much each independent network\ncontributes to the output, using some kind of final head)\n- CIL/DIL mix, by retraining on old classes while adding new ones(issue: old digits will be higher weighted)\n\n'

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

# Base MNIST NN
class MLP(nn.Module):
  def __init__(self, hidden1, hidden2, hidden3, output):
    super().__init__()
    self.fc1 = nn.Linear(28*28, hidden1)
    self.relu1 = nn.ReLU()
    self.fc2 = nn.Linear(hidden1, hidden2)
    self.relu2 = nn.ReLU()
    self.fc3 = nn.Linear(hidden2, hidden3)
    self.relu3 = nn.ReLU()
    self.fc4 = nn.Linear(hidden3, output)
  def forward(self, x):
    out = self.relu1(self.fc1(x))
    out = self.relu2(self.fc2(out))
    out = self.relu3(self.fc3(out))
    out = self.fc4(out)
    return out
  
# Base MNIST NN

transform = transforms.ToTensor()



mnist_train = MNIST(root='./data', train=True, download=True, transform = transforms.ToTensor())
mnist_test = MNIST(root='./data', train=False, download=True, transform = transforms.ToTensor())

layer1 = 256
layer2 = 512
layer3 = 128

model = MLP(layer1, layer2, layer3, 10).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

batch_size = 300
batched_train = torch.utils.data.DataLoader(mnist_train, batch_size=batch_size, shuffle=True)
batched_test = torch.utils.data.DataLoader(mnist_test, batch_size=batch_size, shuffle=False)

loss_func = nn.CrossEntropyLoss()

Using cuda device


In [3]:
print("Starting Training")
model.train()
epochs = 20
for epoch in range(epochs):
  pbar = tqdm(batched_train, desc=f"Epoch {epoch+1}/{epochs}", leave=False)

  for images, labels in pbar:
    images = images.to(device)
    labels = labels.to(device)
    images = torch.flatten(images, start_dim=1)
    optimizer.zero_grad()
    output = model(images)
    loss = loss_func(output, labels)
    loss.backward()
    optimizer.step()
    pbar.set_postfix(loss=loss.item())

Starting Training


In [4]:
model.eval()
total_loss = 0
total = 0
correct = 0
with torch.no_grad():
  correct = 0
  for images, labels in batched_test:
    images = images.to(device)
    labels = labels.to(device)
    images = torch.flatten(images, start_dim=1)
    logits = model(images)
    loss = loss_func(logits, labels)

    total_loss += loss.item() * images.size(0)
    preds = logits.argmax(dim=1)
    correct += (preds == labels).sum().item()
    total += images.size(0)


print(f"Average Loss: {total_loss/correct}, Accuracy: {correct/total}")

Average Loss: 0.09090156164354805, Accuracy: 0.9825


In [6]:
# To return to the model

torch.save(model.state_dict(), "mnist_mlp.pt")

In [3]:
# Re load the pretrained model if needed

model = MLP(layer1, layer2, layer3, 10).to(device)
model.load_state_dict(torch.load("mnist_mlp.pt"))
model.eval()

MLP(
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=256, out_features=512, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=512, out_features=128, bias=True)
  (relu3): ReLU()
  (fc4): Linear(in_features=128, out_features=10, bias=True)
)

In [4]:
def expand_classifier(model, new_size):
  old_fc = model.fc4
  in_features = old_fc.in_features
  old_out = old_fc.out_features
  newlayer = nn.Linear(in_features, old_out + new_size).to(device)

  with torch.no_grad():
    newlayer.weight[:old_out].copy_(old_fc.weight)
    newlayer.bias[:old_out].copy_(old_fc.bias)
  model.fc4 = newlayer
  return model

In [5]:
# Base pure CIL

teacher = copy.deepcopy(model).to(device)
for p in teacher.parameters():
    p.requires_grad_(False)
teacher.eval()

Edata_train = EMNIST(root="./data", split="balanced", train=True, download=True, transform=transforms.ToTensor())
letter_indices = torch.where(Edata_train.targets >= 10)[0]
letters_only_train = Subset(Edata_train, letter_indices)

batch_size = 1000
new_train = torch.utils.data.DataLoader(letters_only_train, batch_size=batch_size, shuffle=True)

Edata_test = EMNIST(root="./data", split="balanced", train=False, download=True, transform=transforms.ToTensor())
letter_indices = torch.where(Edata_test.targets >= 10)[0]
letters_only_test = Subset(Edata_test, letter_indices)

new_test = torch.utils.data.DataLoader(letters_only_test, batch_size=batch_size, shuffle=False)
overall_test = torch.utils.data.DataLoader(Edata_test, batch_size=batch_size, shuffle=False)


num_old = 10
num_total = 48
num_newclasses = num_total - num_old

student = expand_classifier(model, num_newclasses)

def calc_accuracies(model, data1, data2, data3):
    model.eval()
    with torch.no_grad():
        correct, total = 0, 0
        for x, y in data1:
            x, y = x.to(device), y.to(device)
            x = torch.flatten(x, start_dim=1)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.numel()
        print("(New Task)Letters acc:", correct / total)
        correct, total = 0, 0
        for x, y in data2:
            x, y = x.to(device), y.to(device)
            x = torch.flatten(x, start_dim=1)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            # for logit, label in zip(logits, y):
            #   print(f"Logit = {logit.argmax()}, Label = {label}")
            total += y.numel()
        print("(Old Task)Numbers acc:", correct / total)
        correct, total = 0, 0
        for x, y in data3:
            x, y = x.to(device), y.to(device)
            x = torch.flatten(x, start_dim=1)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.numel()
        print("(Combined)Total acc:", correct / total)



In [ ]:
# Begin new training
student = student.to(device)
student.train()
optimizer = torch.optim.Adam(student.parameters(), lr=5e-5)
epochs = 1000

T = 4 # temperature
alpha_kd = .9 # weight of distillation/remembering

for epoch in range(epochs):
  if epoch%10 == 0 or epoch < 5: calc_accuracies(student, new_test, batched_test, overall_test)

  if epoch > epochs // 3: alpha_kd = .7
  else: alpha_kd = .9

  pbar = tqdm(new_train, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
  for images, labels in pbar:
    images = images.to(device)
    labels = labels.to(device)
    images = torch.flatten(images, start_dim=1)

    logits_stud = student(images)
    loss_stud_new = F.cross_entropy(logits_stud[:, 10:], labels - 10)

    with torch.no_grad():
      logits_teach = teacher(images)
      loss_teach = F.softmax(logits_teach/T, dim=1)
    loss_stud_old = F.log_softmax(logits_stud[:, :10]/T, dim=1)
    loss_old = F.kl_div(loss_stud_old, loss_teach, reduction="batchmean") * (T**2)

    loss = (1-alpha_kd) * loss_stud_new + alpha_kd * loss_old
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    pbar.set_postfix(loss=loss.item())



(New Task)Letters acc: 0.0002702702702702703
(Old Task)Numbers acc: 0.9826
(Combined)Total acc: 0.032606382978723404


(New Task)Letters acc: 0.0002702702702702703
(Old Task)Numbers acc: 0.9827
(Combined)Total acc: 0.0325531914893617


(New Task)Letters acc: 0.00033783783783783786
(Old Task)Numbers acc: 0.9824
(Combined)Total acc: 0.032606382978723404


(New Task)Letters acc: 0.0004054054054054054
(Old Task)Numbers acc: 0.9824
(Combined)Total acc: 0.03276595744680851


(New Task)Letters acc: 0.0007432432432432432
(Old Task)Numbers acc: 0.9824
(Combined)Total acc: 0.03292553191489362


...

(New Task)Letters acc: 0.2656081081081081
(Old Task)Numbers acc: 0.9811
(Combined)Total acc: 0.23882978723404255


(New Task)Letters acc: 0.2699324324324324
(Old Task)Numbers acc: 0.9809
(Combined)Total acc: 0.2422872340425532


(New Task)Letters acc: 0.2727702702702703
(Old Task)Numbers acc: 0.981
(Combined)Total acc: 0.2444148936170213


(New Task)Letters acc: 0.2764864864864865
(Old Task)Numbers acc: 0.981
(Combined)Total acc: 0.2473404255319149


Epoch 214/1000:  93%|█████████▎| 83/89 [00:09<00:00,  8.84it/s, loss=0.0437]

In [9]:
torch.save(student.state_dict(), "emnist_mlp.pt")

In [ ]:
student = MLP(layer1, layer2, layer3, 48).to(device)
student.load_state_dict(torch.load("emnist_mlp.pt"))
student.eval()

In [ ]:

calc_accuracies(student, new_test, batched_test, overall_test)

(New Task)Letters acc: 0.794527027027027
(Old Task)Numbers acc: 0.6085
(Combined)Total acc: 0.6393617021276595


In [10]:
student.eval()
total_loss = 0
total = 0
correct = 0
with torch.no_grad():
  correct = 0
  for images, labels in batched_test:
    images = images.to(device)
    labels = labels.to(device)
    images = torch.flatten(images, start_dim=1)
    logits = student(images)
    loss = loss_func(logits, labels)

    total_loss += loss.item() * images.size(0)
    preds = logits.argmax(dim=1)
    correct += (preds == labels).sum().item()
    total += images.size(0)

print(f"Average Loss: {total_loss/correct}, Accuracy: {correct/total}")

Average Loss: 10.330389425611536, Accuracy: 0.6085
